# 44. Train / Validation Loop 구현

이 노트북은 semantic segmentation 모델 학습 loop의 기본 구조를 다룹니다.

이번 노트북의 목표는 다음과 같습니다.

- logits와 mask shape가 loss에 어떻게 연결되는지 확인합니다.
- train loop와 validation loop를 분리합니다.
- small batch overfit 테스트의 의미를 이해합니다.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

torch.manual_seed(1)
np.random.seed(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
class SyntheticSegmentationDataset(Dataset):
    def __init__(self, length=48, size=64):
        self.length = length
        self.size = size

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        h = w = self.size
        yy, xx = np.mgrid[:h, :w]
        image = np.zeros((3, h, w), dtype=np.float32) + 0.45
        mask = np.zeros((h, w), dtype=np.int64)

        circle = (xx - (18 + idx % 20)) ** 2 + (yy - 28) ** 2 < 12 ** 2
        rect = (xx > 34) & (xx < 55) & (yy > 18 + idx % 8) & (yy < 48)

        image[0, circle] = 0.9
        image[1, rect] = 0.85
        mask[circle] = 1
        mask[rect] = 2
        image += np.random.normal(0, 0.04, image.shape).astype(np.float32)
        return torch.from_numpy(np.clip(image, 0, 1)), torch.from_numpy(mask)


dataset = SyntheticSegmentationDataset()
train_set, val_set = random_split(dataset, [40, 8], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader = DataLoader(val_set, batch_size=8)

## 44-1. 작은 segmentation 모델

실제 프로젝트에서는 이 모델 자리에 SegFormer가 들어갑니다. 여기서는 loop 구조를 빠르게 확인하기 위해 작은 CNN을 사용합니다.

In [ ]:
class TinySegModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, num_classes, 1),
        )

    def forward(self, x):
        return self.net(x)


model = TinySegModel(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 44-2. 한 batch에서 shape 확인

`CrossEntropyLoss`는 segmentation에서 다음 shape를 기대합니다.

```text
logits: B, C, H, W
target: B, H, W
```

In [ ]:
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)
logits = model(images)
loss = criterion(logits, masks)

print("images:", images.shape)
print("masks:", masks.shape)
print("logits:", logits.shape)
print("loss:", float(loss))

## 44-3. Train loop와 validation loop

In [ ]:
def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss = 0.0

    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)

        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


for epoch in range(5):
    train_loss = run_epoch(model, train_loader, train=True)
    val_loss = run_epoch(model, val_loader, train=False)
    print(f"epoch {epoch + 1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

## 44-4. Small batch overfit 테스트

실전 구현이 의심될 때는 아주 작은 batch 하나에 overfit되는지 먼저 확인합니다. 이것도 안 되면 데이터, label, loss, model forward 중 하나가 잘못되었을 가능성이 큽니다.

## 정리

- segmentation train loop의 핵심은 logits와 mask shape를 맞추는 것입니다.
- train mode와 validation mode를 분리해야 합니다.
- 다음 노트북 `45_Loss_Function과_Class_Imbalance.ipynb`에서는 class imbalance와 loss 선택을 다룹니다.